# Model Interpretability and Feature AnalysisThis notebook explains how the trained models make predictions, using feature importance analysis and SHAP (SHapley Additive exPlanations) values. Understanding model decisions is crucial in healthcare applications.

In [ ]:
from pathlib import Pathimport sysimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport joblibimport shapsys.path.append(str(Path.cwd().parent / 'src'))from data_loader import load_cdc_diabetes_datasns.set_style('whitegrid')plt.rcParams.update({'figure.figsize': (12, 7), 'figure.dpi': 100})MODELS_DIR = Path.cwd().parent / 'models'OUTPUT_DIR = Path.cwd().parent / 'images'OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load data and best model

In [ ]:
X, y, df = load_cdc_diabetes_data()print(f"Dataset shape: {df.shape}")print(f"Features: {X.shape[1]}")print(f"Target distribution:\n{y.value_counts()}")# Load best model from previous notebookbest_model_path = MODELS_DIR / "best_model.joblib"if best_model_path.exists():    best_model = joblib.load(best_model_path)    print(f"\n✅ Loaded best model from {best_model_path}")else:    print("⚠️  Best model not found. Training a new model...")    from sklearn.model_selection import train_test_split    from sklearn.pipeline import Pipeline    from sklearn.compose import ColumnTransformer    from sklearn.impute import SimpleImputer    from sklearn.preprocessing import StandardScaler    from sklearn.ensemble import GradientBoostingClassifier        numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()    categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()        numeric_transformer = Pipeline([        ("imputer", SimpleImputer(strategy="median")),        ("scaler", StandardScaler()),    ])        categorical_transformer = Pipeline([        ("imputer", SimpleImputer(strategy="most_frequent"))    ])        preprocessor = ColumnTransformer(        transformers=[            ("num", numeric_transformer, numeric_features),            ("cat", categorical_transformer, categorical_features),        ],        remainder="passthrough"    )        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)        best_model = Pipeline([        ("preprocessor", preprocessor),        ("classifier", GradientBoostingClassifier(n_estimators=100, random_state=42)),    ])        best_model.fit(X_train, y_train)    print(f"Model trained! Accuracy: {best_model.score(X_test, y_test):.4f}")

## Feature Importance AnalysisTree-based models (like Gradient Boosting) provide feature importance scores based on how much each feature contributes to reducing prediction errors.

In [ ]:
# Extract the classifier from the pipelineclassifier = best_model.named_steps["classifier"]if hasattr(classifier, "feature_importances_"):    feature_importance = classifier.feature_importances_    feature_names = X.columns        # Create a dataframe for better visualization    importance_df = pd.DataFrame({        "Feature": feature_names,        "Importance": feature_importance    }).sort_values("Importance", ascending=False)        print("Top 10 Most Important Features:")    print(importance_df.head(10))        # Plot feature importance    plt.figure(figsize=(10, 6))    sns.barplot(data=importance_df.head(10), x="Importance", y="Feature", palette="viridis")    plt.title("Top 10 Feature Importance (Gradient Boosting)", fontsize=14, fontweight="bold")    plt.xlabel("Importance Score")    plt.tight_layout()    plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=300, bbox_inches="tight")    plt.show()else:    print("Model does not have feature_importances_ attribute")

## SHAP (SHapley Additive exPlanations) ValuesSHAP values provide a theoretically sound way to interpret individual predictions. Each feature is assigned a value that represents its contribution to pushing the prediction away from the base value.

In [ ]:
# Prepare data for SHAPfrom sklearn.model_selection import train_test_splitX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)# Create SHAP explainer for the classifiertry:    explainer = shap.TreeExplainer(classifier)    shap_values = explainer.shap_values(X_test)        print("✅ SHAP values computed")    print(f"Shape of SHAP values: {np.array(shap_values).shape}")        # For binary classification, shap_values might be a list with 2 elements (one for each class)    # We take the values for the positive class (diabetes=1)    if isinstance(shap_values, list):        shap_vals = shap_values[1]    else:        shap_vals = shap_values        # Summary plot    plt.figure(figsize=(10, 8))    shap.summary_plot(shap_vals, X_test, plot_type="bar", show=False)    plt.title("SHAP Summary (Mean Absolute SHAP values)")    plt.tight_layout()    plt.savefig(OUTPUT_DIR / "shap_summary_bar.png", dpi=300, bbox_inches="tight")    plt.show()        print("\n✅ SHAP summary plot saved")except Exception as e:    print(f"Note: SHAP analysis could not be completed: {e}")

## Individual Prediction ExplanationFor a healthcare application, it is important to understand why the model makes a specific prediction for an individual patient.

In [ ]:
# Select a sample to explainsample_idx = 0  # First test samplesample = X_test.iloc[[sample_idx]]sample_shap_values = shap_vals[sample_idx]# Get predictionprediction = best_model.predict(sample)[0]prediction_proba = best_model.predict_proba(sample)[0]print(f"Sample {sample_idx} Characteristics:")print(sample.T)print(f"\nPredicted Class: {prediction} (Diabetes: {"Yes" if prediction == 1 else "No"})")print(f"Prediction Probability: {prediction_proba}")print(f"\nTop Contributing Features to this Prediction:")# Get top contributing featuresfeature_impact = pd.DataFrame({    "Feature": X_test.columns,    "SHAP_Value": sample_shap_values,    "Feature_Value": sample.values.flatten()}).sort_values("SHAP_Value", ascending=False, key=abs)print(feature_impact.head(5))

## Model Reliability and ConfidenceIn healthcare, not only should we know what a model predicts, but also how confident we should be in that prediction.

In [ ]:
# Calculate prediction confidenceprobabilities = best_model.predict_proba(X_test)confidence = np.max(probabilities, axis=1)print(f"Mean Prediction Confidence: {confidence.mean():.4f}")print(f"Std Dev: {confidence.std():.4f}")print(f"Min: {confidence.min():.4f}")print(f"Max: {confidence.max():.4f}")# Visualize confidence distributionplt.figure(figsize=(10, 6))plt.hist(confidence, bins=30, color="steelblue", edgecolor="black", alpha=0.7)plt.xlabel("Prediction Confidence", fontsize=12)plt.ylabel("Frequency", fontsize=12)plt.title("Distribution of Model Prediction Confidence", fontsize=14, fontweight="bold")plt.axvline(confidence.mean(), color="red", linestyle="--", linewidth=2, label=f"Mean: {confidence.mean():.3f}")plt.legend()plt.tight_layout()plt.savefig(OUTPUT_DIR / "prediction_confidence.png", dpi=300, bbox_inches="tight")plt.show()

## SummaryThis notebook demonstrated:- **Feature Importance**: Understanding which features drive model predictions- **SHAP Values**: A theoretically grounded approach to feature attribution- **Individual Explanations**: Why the model makes a specific prediction- **Prediction Confidence**: Assessing how certain the model isThese interpretability tools are essential for building trust in ML models for healthcare applications.